# BIOT 6900 · Module 2 — Multi-Omics Target Identification & Validation
### Week 2 · Computational Lab 2 · Project Notebook

In [8]:
import os
import numpy as np
import pandas as pd
from scipy import stats

# Default scoring weights for this course: equal across the three layers.
# You MAY change these in Part 3 — but if you do, you must justify it in your report. - NEED TO CHANGE THIS IN CAPSTONE PROJECT
EQUAL_WEIGHTS = {"transcriptomic": 1/3, "proteomic": 1/3, "genomic": 1/3}

In [9]:
def rank_percentile(series: pd.Series) -> pd.Series:
    """Normalize any score to [0, 1] by rank. Robust to outliers and scale differences."""
    return series.rank(method="average", pct=True)


def multi_evidence_score(df: pd.DataFrame, cols, weights) -> pd.Series:
    """Weighted sum of rank-percentile-normalized layer scores."""
    normed = pd.DataFrame({c: rank_percentile(df[c]) for c in cols})
    return sum(weights[c] * normed[c] for c in cols)

---
# Part 3 — My project: Pancreatic ductal adenocarcinoma (PDAC)

I am using the same gene-level workflow from the assignment, but on PDAC.

- RNA: tumor vs normal effect + p-value
- Protein: tumor vs normal effect + p-value
- Genomics: somatic mutation frequency as association strength

### 3.1 — Load and check the three PDAC files

In [10]:
from pathlib import Path

data_dir = Path("../data/pdac")
if not (data_dir / "pdac_transcriptomics.tsv").exists():
    data_dir = Path(".")

tx = pd.read_csv(data_dir / "pdac_transcriptomics.tsv", sep="\t")
pr = pd.read_csv(data_dir / "pdac_proteomics.tsv", sep="\t")
gw = pd.read_csv(data_dir / "pdac_genomics.tsv", sep="\t")

for name, d in {"RNA": tx, "Protein": pr, "Genomics": gw}.items():
    print(name, d.shape, d.columns.tolist())

RNA (28057, 3) ['gene', 'effect', 'pval']
Protein (11662, 3) ['gene', 'effect', 'pval']
Genomics (4424, 2) ['gene', 'association']


### 3.2 — Join all three layers by gene

In [11]:
for d in (tx, pr, gw):
    d["gene"] = d["gene"].astype(str).str.strip().str.upper()

tx = tx.rename(columns={"effect": "rna_effect", "pval": "rna_p"})
pr = pr.rename(columns={"effect": "prot_effect", "pval": "prot_p"})

df = tx.merge(pr, on="gene").merge(gw, on="gene")

print(f"Genes surviving the 3-way join: {len(df)}")
df.head()

Genes surviving the 3-way join: 2539


,gene,rna_effect,rna_p,prot_effect,prot_p,association
0,A1BG,-0.479548,1.250690e-02,0.303070,2.094465e-04,0.007143
1,A2ML1,3.030196,2.441368e-12,0.884157,2.971118e-08,0.007143
2,A4GNT,-0.688471,6.228432e-02,-0.137812,4.689057e-01,0.007143
3,AAGAB,0.321240,4.493614e-07,0.179743,6.447061e-04,0.007143
4,AARS,-0.127723,1.951716e-01,-0.613124,1.002847e-13,0.007143


### 3.3 — RNA/protein direction agreement

In [12]:
df["concordant"] = np.sign(df["rna_effect"]) == np.sign(df["prot_effect"])

print(f"Concordant: {df['concordant'].sum()} / {len(df)} "
      f"({df['concordant'].mean():.1%})")

Concordant: 1650 / 2539 (65.0%)


### 3.4 — Multi-evidence score

I used absolute RNA/protein effect size and mutation frequency. I kept equal weights for this first pass, same as the class example.

In [13]:
df["transcriptomic"] = df["rna_effect"].abs()
df["proteomic"] = df["prot_effect"].abs()
df["genomic"] = df["association"]

df["score"] = multi_evidence_score(
    df,
    ["transcriptomic", "proteomic", "genomic"],
    EQUAL_WEIGHTS
)

### 3.5 — Rank and export

In [14]:
ranked = df.sort_values("score", ascending=False).reset_index(drop=True)
ranked["rank"] = ranked.index + 1

show = ["rank", "gene", "rna_effect", "rna_p", "prot_effect",
        "prot_p", "association", "concordant", "score"]

print("Top 15:")
display(ranked[show].head(15))

check_genes = ["KRAS", "TP53", "CDKN2A"] #I know from experience that KRAS (G12C, G12D... mutations) is an important biomarker so checking to see what rank it has
print("\nPDAC genes I wanted to check:")
display(ranked[ranked["gene"].isin(check_genes)][show])

print("\nTop discordant genes:")
display(ranked[~ranked["concordant"]][show].head(10))

ranked.to_csv("targets_pdac.csv", index=False)
print("\nSaved targets_pdac.csv")

Top 15:


,rank,gene,rna_effect,rna_p,prot_effect,prot_p,association,concordant,score
0,1,COL11A1,2.787325,5.684449e-06,1.448051,4.302139e-29,0.028571,True,0.984230
1,2,MUC5B,2.071322,5.008643e-06,1.224256,8.306296e-18,0.042857,True,0.976274
2,3,TNS4,4.351478,3.304188e-10,1.233976,8.598678e-22,0.021429,True,0.974962
3,4,ERBB4,-2.565261,3.790142e-15,1.328604,9.735195e-03,0.021429,False,0.972867
4,5,PLIN4,-3.068218,1.254984e-07,-1.197169,3.684043e-08,0.021429,True,0.972464
5,6,MUC16,3.162246,1.300220e-06,0.922216,1.512537e-08,0.078571,True,0.971442
6,7,FN1,1.826157,9.992815e-06,1.263289,1.485067e-32,0.028571,True,0.971223
7,8,POSTN,1.743142,5.037925e-04,1.353190,1.520684e-29,0.028571,True,0.970965
8,9,NOS1,-1.915787,4.607875e-06,-1.252302,1.529237e-05,0.021429,True,0.964461
9,10,MUC5AC,3.750809,8.660646e-04,0.814858,8.955681e-08,0.050000,True,0.961767



PDAC genes I wanted to check:


,rank,gene,rna_effect,rna_p,prot_effect,prot_p,association,concordant,score
35,36,CDKN2A,1.204299,6.146961e-04,0.642629,0.033929,0.207143,True,0.914645
277,278,KRAS,0.632832,2.848552e-08,0.195806,0.000123,0.964286,True,0.725972
911,912,TP53,0.034391,5.552805e-01,0.333604,0.012389,0.750000,True,0.553770



Top discordant genes:


,rank,gene,rna_effect,rna_p,prot_effect,prot_p,association,concordant,score
3,4,ERBB4,-2.565261,3.790142e-15,1.328604,9.735195e-03,0.021429,False,0.972867
47,48,ACTN2,-1.022766,2.126394e-06,0.947958,2.797237e-12,0.014286,False,0.890772
48,49,LRP1B,-1.319702,8.595829e-10,0.519976,2.456266e-02,0.057143,False,0.889228
55,56,LPA,-2.403010,7.885824e-06,0.462555,9.775410e-03,0.021429,False,0.881245
60,61,TF,-1.452412,2.223056e-07,0.638961,2.093351e-10,0.014286,False,0.877161
72,73,FBLN1,-0.684329,4.608781e-03,0.902747,2.564244e-25,0.014286,False,0.862798
77,78,BAAT,-1.429300,9.029701e-06,0.560057,2.336404e-01,0.014286,False,0.857548
80,81,ASPM,2.278870,6.048612e-11,-0.468939,4.171212e-02,0.014286,False,0.850005
81,82,PCDHB6,-0.580394,1.023166e-04,0.621654,2.468220e-05,0.021429,False,0.849316
83,84,ADGRV1,-2.602030,6.168645e-15,0.319792,3.380941e-01,0.042857,False,0.846454



Saved targets_pdac.csv


### 3.6 — My interpretation

**Weighting.** I kept equal weights for this first pass because I did not want to decide one layer is more important without a reason. KRAS has very high mutation frequency but is not near the top because its RNA/protein effects are smaller. For the capstone I may test a genomics-heavy score, but I am keeping the class default here.

**Top targets.** COL11A1 ranked first, followed by MUC5B and TNS4. CDKN2A ranked 36, KRAS 278, SMAD4 458 and TP53 912. So this score is not just finding common mutation drivers; it rewards genes that have evidence across the three layers.

**Discordance.** ERBB4 ranked 4 but RNA went down while protein went up. I would not call it a target just from this result. This could be post-transcriptional regulation/protein stability, or some other reason I should check with biology/pathology experts. It needs follow-up.

**Limitation.** These are gene-level summaries, so I cannot make patient-level claims from this analysis. Also the genomic value here is somatic mutation frequency, not a GWAS p-value. Equal weights and percentile ranks are useful for a first ranking, but the scoring choice clearly changes which genes come to the top.